![DB Academy](./Includes/images/db-academy.png)

# Reverse ETL — Moving Lakehouse Data into Lakebase

---

Lets discuss how to bring curated analytics data from the Databricks Lakehouse into a Lakebase Postgres database — making it available to low-latency applications without duplicating your analytical pipelines.

## Learning Objectives
By the end of this lecture, you will understand:

1. **Reverse ETL** - what it is, why it matters, and how Lakebase simplifies this process! 
2. **Synced Tables** - what they are, how to create them, and best practices to have in mind while using them.
3. **Sync Modes** - (Snapshot, Triggered, Continuous) and when to use each.

## A. Reverse ETL

![reverse_etl_wo_lakebase.png](Includes/images/reverse_etl/reverse_etl_w_o_lakebase.png)

### What is it?

Traditional ETL moves raw data *into* a lakehouse for analysis. **Reverse ETL is the return journey** — it takes curated, analytics-grade data *out* of the lakehouse and pushes it into an OLTP engine, where applications can query it for their low latency use cases.

Most teams face mulitiple challenges (management of custom pipelines as data changes/grows over time, inconsistent governance models etc) when handling this process on their own. These challenges can create friction for both developers and the business, slowing down efforts to reliably activate data and deliver intelligent, real-time applications.

**Lakebase Simplifies this process**

<br>

![reverse_etl_w_lakebase.png](Includes/images/reverse_etl/reverse_etl_w_lakebase.png)

When using Lakebase as part of the Data Intelligence platform, sync to the operational engine is **managed by Databricks**. This allows developers to focus their time on building their real time apps and less time on managing infrastructure.

## B. Synced Tables

![synced_tables.png](Includes/images/reverse_etl/synced_tables.png)

### What are they?

A **synced table** is a managed, read-only copy of a Unity Catalog table inside Lakebase. Creating one links 2 objects:

| Object | Location | Purpose |
|--------|----------|---------|
| **Unity Catalog table** | Lakehouse (Delta) | Source of truth; managed by the sync pipeline |
| **Postgres table** | Lakebase database | App-facing replica; automatically kept in sync |

For example, syncing `analytics.gold.user_profiles` creates a Postgres table you query as:
```sql
SELECT * FROM "gold"."user_profiles_synced";
```

> ⚠️ Synced tables in Postgres are **read-only**. Only create indexes or drop the table (after removing it from Unity Catalog). Direct writes risk breaking sync integrity.

---
### B1. Sync Modes

### What are the sync modes?

Lakebase offers three sync modes for syncing data from the Lakehouse into Lakebase. Each comes with its own unique **latency**, **cost**, and **source requirements** trade off:

| Mode | How it works | Latency | Cost | CDF Required? |
|------|-------------|---------|------|---------------|
| **Snapshot** | Replaces the entire Postgres table on every refresh | Minutes (batch) | Low | No |
| **Triggered** | Applies only the changes since the last sync, on a schedule | Minutes (scheduled) | Medium | Yes |
| **Continuous** | Streams changes into Postgres in near real-time | Seconds | High | Yes |

#### Throughput reference

| Mode | Rows/second per Capacity Unit |
|------|-------------------------------|
| Snapshot | ~2,000 |
| Triggered / Continuous | ~150 |

> Snapshot is ~13× faster per CU because it bulk-loads data without tracking individual row changes.

---

### When to use which?

#### 🟦 Snapshot — Use when freshness can wait and data changes heavily

- Full table replacement on every run — simple and robust
- No CDF requirement on the source table
- **Best choice when more than ~10% of source rows change between syncs** — at that threshold, tracking individual changes costs more than a full reload

```
Good for:  Initial data loads, daily batch refreshes, tables without CDF,
           tables with high churn (e.g. aggregated metrics that are fully recomputed)
```

#### 🟨 Triggered — Use when you need periodic freshness without continuous cost

- Syncs only the delta (changed rows) on a schedule you define (e.g. every hour)
- Much cheaper than Continuous for workloads where second-level latency isn't required
- Requires CDF enabled on the source table

```
Good for:  Dashboard backing data, recommendation models refreshed hourly,
           reporting tables, any use case where "fresh within the last N minutes" is sufficient
```

#### 🟥 Continuous — Use when your app needs near real-time data

- Streams changes within seconds of them landing in the Lakehouse
- Highest compute cost — runs a persistent streaming job
- Requires CDF enabled on the source table

```
Good for:  Live inventory checks, fraud detection, personalisation at request time,
           any user-facing feature where stale data causes visible errors
```

---

### Decision guide

```
Does your app need data fresh within seconds?
  ├── Yes  →  Continuous
  └── No
       │
       Does more than ~10% of the source table change per sync cycle?
         ├── Yes  →  Snapshot  (full reload is more efficient)
         └── No   →  Triggered  (delta sync is cheaper)
```



### B2. How to create a Synced Table in a Lakebase Project

#### Prerequisites

- An active Lakebase project with at least one branch and endpoint
- `USE_SCHEMA` and `CREATE_TABLE` permissions on the target schema
- For Triggered or Continuous modes: Change Data Feed (CDF) enabled on the source table

#### Step 1 — Create the Synced Table (UI)

<br>

```
Catalog  →  Select your Unity Catalog table
         →  Create  →  Synced table
         →  Fill in:
              • Synced table name        (alphanumeric + underscores only)
              • Compute type             "Lakebase Serverless (Autoscaling)"
              • Sync mode                Snapshot | Triggered | Continuous
              • Project, branch, database
              • Primary key              (auto-detected from source)
         →  Create
```

#### Step 1.1 — Enable Change Data Feed (Triggered / Continuous only)

<br>

```sql
ALTER TABLE analytics.gold.user_profiles
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);
```

#### Step 2 — Monitor Sync Status

Track progress in **Catalog → Overview tab**. Use **"Sync now"** to trigger a manual refresh at any time.

#### Operational limits

| Limit | Value |
|-------|-------|
| Max total data across all synced tables | 8 TB |
| Recommended max per refreshable table | 1 TB |
| DB connections per synced table | Up to 16 |
| Schema evolution support | Additive changes only (Triggered / Continuous) |

### B3. Data Types & Incompatibility

Lakebase automatically maps Unity Catalog types to their Postgres equivalents:

| Unity Catalog Type | Postgres Type | Notes |
|--------------------|--------------|-------|
| `BIGINT` | `BIGINT` | |
| `INT`, `SMALLINT`, `TINYINT` | `INTEGER` variants | |
| `STRING` | `TEXT` | Null bytes (0x00) not allowed — see below |
| `BOOLEAN` | `BOOLEAN` | |
| `DATE` | `DATE` | |
| `TIMESTAMP` | `TIMESTAMP WITHOUT TIME ZONE` | |
| `TIMESTAMP_NTZ` | `TIMESTAMP WITH TIME ZONE` | |
| `DECIMAL(p,s)` | `NUMERIC(p,s)` | |
| `BINARY` | `BYTEA` | |
| `ARRAY`, `MAP`, `STRUCT` | `JSONB` | Nested types serialised as JSON |
| `GEOGRAPHY`, `GEOMETRY` | ❌ Not supported | |
| `VARIANT`, `OBJECT` | ❌ Not supported | |

#### How to Handle Invalid Data Types

The most common incompatibility is the **null byte (`0x00`)** — valid in Delta/Unity Catalog strings but rejected by Postgres `TEXT` and `JSONB`.

**Option 1 — Strip null bytes at source (recommended)**

Clean the column in your Unity Catalog table before syncing:

<br>

```sql
-- Remove null bytes from a STRING column
SELECT REPLACE(column_name, CAST(CHAR(0) AS STRING), '') AS cleaned_column
FROM analytics.gold.user_profiles;
```

**Option 2 — Convert to BINARY**

If you need to preserve the raw bytes exactly (e.g. for downstream decoding), cast the column to `BINARY` — it maps to Postgres `BYTEA` which accepts any byte sequence:

<br>

```sql
-- Cast to BINARY to bypass TEXT restrictions
SELECT CAST(column_name AS BINARY) AS raw_column
FROM analytics.gold.user_profiles;
```

> 💡 Always clean or cast problematic columns **in the source Unity Catalog table**, not in Postgres. The sync pipeline reads from the source — transformations applied after the fact won't propagate.